# Lesson 10 &mdash; Convolutional Networks, and Course Synthesis

Self-assessment, and the last of the course. No code: every answer is a
sentence, a short derivation, or a judgement.

The numbers quoted come from the lesson's handout and its three notebooks.
The problem throughout is Meridian Instruments' silicon dies, photographed at
24&times;24 pixels &mdash; 576 pixels &mdash; and graded pass or fail by a
station that records the wrong verdict for a designed 2% of units. That caps
every accuracy here: 0.9800 on the 2,000-image test set used for most of the
lesson, 0.9847 on the separate set used for the moved-defect and transfer
experiments, because each set realises its own rate. Read every score against
the ceiling of the set it was measured on, never against 1.0.

Several questions ask you to *derive* a count or an output size rather than
recall it, and those are the ones worth your time. Four more ask about
results that contradict a slogan you have probably already absorbed &mdash;
about permutation, about transfer learning, and about what classical methods
can do with an image. The last section reaches back over all ten weeks.


## Part 1 &mdash; Two properties of a wafer, and why a dense layer is the wrong shape


**1. Name the two properties of a wafer defect that decide the whole
architecture, and say which design choice each one motivates.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>The defect is local.</b> A scratch is about seven pixels long and a particle is a small bright blob; the other 570-odd pixels of the die are irrelevant to the verdict. This motivates a <b>small kernel</b> &mdash; a unit that looks at a 3&times;3 window rather than at all 576 pixels at once.</li>
        <li><b>The defect means the same thing wherever it lands.</b> A scratch top-left and the same scratch bottom-right are the same event, graded the same way. This motivates <b>weight sharing</b> &mdash; applying one kernel at every position &mdash; and, at the end, a <b>global maximum</b>, which asks only whether a strong response occurred anywhere.</li>
        <li>A third property lurks in the images and rules out the obvious detector: film thickness varies smoothly across a die, so <b>absolute brightness says nothing</b>. A defect is dark or bright <i>relative to its immediate surroundings</i>. That is what question 9's zero-sum kernel is for.</li>
    </ul>
    </p>
</details>


**2. Verify the two headline parameter counts: 147,712 for a dense layer
of 256 units on these images, and 80 for a convolutional layer of eight 3&times;3
kernels. Show the arithmetic.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Dense.</b> Each unit connects to every one of the 576 pixels and has its own bias: <code>576 &times; 256 = 147,456</code> weights, plus <code>256</code> biases, giving <code>147,712</code>. Widen it to 1,024 units and you get <code>576 &times; 1024 + 1024 = 590,848</code>.</li>
        <li><b>Convolutional.</b> Each kernel is nine weights and one bias, whatever the size of the image it is slid over: <code>8 &times; 9 + 8 = 80</code>. The image size does not appear in the count at all, which is the first sign that something structural has changed.</li>
        <li>The ratio is <code>147,712 / 80 = 1,846</code>. Keep it in view for question 3, which argues that it is the least interesting fact in the table.</li>
    </ul>
    </p>
</details>


**3. Explain why reading that table as &ldquo;the convolutional layer is
smaller&rdquo; misses the point twice over.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>It is not smaller in output.</b> Eight feature maps of 24&times;24 is <code>4,608</code> numbers, against <code>256</code> from the dense layer. The convolutional layer produces <i>eighteen times more</i> activations; it is a larger layer that happens to be described by fewer weights.</li>
        <li><b>The saving is not the benefit.</b> The 80 weights are few because the <i>same</i> nine numbers are used at every position, and what that buys is not memory but <b>evidence</b>: one example of a scratch anywhere on any die trains the detector for everywhere. The dense layer must be taught position by position, and section 8 of the handout measures the price &mdash; it is still 0.235 short of the ceiling at 8,000 training images.</li>
        <li>Compression is a side effect worth having on a small machine. It is not the argument, and a student who remembers only the ratio has kept the wrong half.</li>
    </ul>
    </p>
</details>


**4. State, in one sentence, what weight sharing buys.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>One labelled example of a pattern teaches the detector for that pattern at <b>every position at once</b>, because there is only one copy of the weights &mdash; so the currency saved is training data, not memory.</li>
    </ul>
    </p>
</details>


**5. The batch used in notebook 01 has 2.55% of its grades wrong against a
designed 2%. Why does the lesson read every score against the ceiling realised
on the scoring set rather than against the design figure of 0.98?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Because the ceiling that constrains a measurement is the one <b>in the data you measured on</b>. The design rate is a property of the generator; the rate that caps a test score is the fraction of <i>that test set</i> whose recorded grade is wrong.</li>
        <li>2.55% is not a bug. Over further seeds the rate averages close to the designed 0.02, and the binomial standard deviation at this sample size is 0.0022, so 0.0255 is a draw about 2.5 standard deviations high &mdash; unusual, entirely possible, and exactly the sort of thing that makes a model look broken when it is not.</li>
        <li>The practical consequence: a model scoring 0.9800 on a set whose ceiling is 0.9800 has <b>no error of its own left</b>. Chasing 0.99 there is chasing the station's mistakes, which is lesson 5's point about knowing what your number can possibly be.</li>
    </ul>
    </p>
</details>


## Part 2 &mdash; The convolution itself


**6. Write down the operation a deep learning library performs when it says
&ldquo;convolution&rdquo;, say how it differs from a true convolution, and explain why
the difference does not matter.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>At each position the layer writes down the weighted sum of the pixels the kernel covers: <code>(I * K)[r, c] = &Sigma;<sub>i</sub> &Sigma;<sub>j</sub> I[r+i, c+j] &middot; K[i, j]</code>, with <code>i, j</code> running from 0 to <code>f&minus;1</code>.</li>
        <li>Strictly this is <b>cross-correlation</b>. A true convolution flips the kernel in both axes first. Every mainstream library computes the expression above and calls it convolution.</li>
        <li>It does not matter because <b>the kernel is learned</b>: whatever weights the flipped version would want, the unflipped one can hold directly. The flip is a relabelling of nine numbers that gradient descent is free to undo. It <i>would</i> matter if you were given a kernel by someone else and had to reproduce their result exactly.</li>
        <li>Notebook 01 implements it in eleven lines and checks it against <code>scipy</code>, an independent implementation by other people. The largest disagreement is <code>8.9 &times; 10<sup>&minus;16</sup></code>, which is floating-point rounding, not an error &mdash; the right order of magnitude to expect from double precision, and the reason it is worth quoting rather than saying &ldquo;they agree&rdquo;.</li>
    </ul>
    </p>
</details>


**7. State the output-size formula and use it on a 24&times;24 image for
(f=3, p=0, s=1), (f=5, p=2, s=1) and (f=3, p=1, s=2). For the last, say exactly
what the floor throws away.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><code>output size = &lfloor;(n + 2p &minus; f) / s&rfloor; + 1</code>, applied independently to each spatial dimension.</li>
        <li><b>f=3, p=0, s=1:</b> <code>(24 + 0 &minus; 3)/1 + 1 = 22</code>. Three pixels are needed for the first window, so the map shrinks by <code>f &minus; 1 = 2</code>.</li>
        <li><b>f=5, p=2, s=1:</b> <code>(24 + 4 &minus; 5)/1 + 1 = 24</code>. Unchanged, because <code>p = (f&minus;1)/2</code> exactly replaces the border the kernel eats.</li>
        <li><b>f=3, p=1, s=2:</b> <code>&lfloor;(24 + 2 &minus; 3)/2&rfloor; + 1 = &lfloor;11.5&rfloor; + 1 = 12</code>. Here the floor does real work. The padded image is 26 columns wide, indices 0 to 25, and windows begin at 0, 2, 4, &hellip;, 22 &mdash; twelve of them, the last covering columns 22&ndash;24. A thirteenth would have to start at 24 and reach column 26, which does not exist, so <b>padded column 25 is never covered by any window</b> and the last incomplete position is simply not taken. Nothing is padded further and no error is raised; the pixel is silently dropped.</li>
    </ul>
    </p>
</details>


**8. You need a layer that leaves a 24&times;24 map the same size, using a
7&times;7 kernel. Give the padding, and give the cheap way to halve a map without
pooling.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Same size:</b> <code>p = (f &minus; 1)/2 = 3</code>. Check it: <code>(24 + 6 &minus; 7)/1 + 1 = 24</code>. The rule <code>p = (f&minus;1)/2</code> works for any odd <code>f</code>, which is why kernels are almost always odd-sized &mdash; an even kernel has no centre and no symmetric padding.</li>
        <li>This is what Keras spells <code>padding="same"</code>, and it is worth knowing the arithmetic behind the string, because a library that computes the padding for you will happily do it asymmetrically for even <code>f</code>.</li>
        <li><b>Halving:</b> set <code>s = 2</code>, which gives 12 as computed in question 7. A strided convolution is the cheap alternative to pooling &mdash; it does the downsampling in the same pass as the filtering, at the price of one number in three being computed from a window that overlaps its neighbours less.</li>
    </ul>
    </p>
</details>


**9. Prove in one line that a kernel whose weights sum to zero is blind to
absolute brightness, and say why this problem needs exactly that.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Take a patch that is <b>constant</b> at value <code>v</code>. The response is <code>&Sigma;<sub>i,j</sub> v &middot; K[i,j] = v &middot; &Sigma;<sub>i,j</sub> K[i,j] = v &middot; 0 = 0</code>, for every <code>v</code>. The constant factors straight out of the sum, so a zero-sum kernel returns nothing on any flat region however bright or dark it is.</li>
        <li>By linearity the same argument covers the general case: write any patch as a constant plus a deviation, and the constant part contributes zero. The kernel reports <b>only local contrast</b> &mdash; how a patch differs from being flat.</li>
        <li>That is precisely the requirement stated in section 1.1. Film thickness varies smoothly across every die, so a detector keyed to &ldquo;is this pixel dark&rdquo; fires on the dark half of every clean die. A zero-sum kernel cannot make that mistake, because the smooth background is locally near-constant and is therefore invisible to it.</li>
        <li>The centre-surround kernel <code>[[&minus;1,&minus;1,&minus;1],[&minus;1,8,&minus;1],[&minus;1,&minus;1,&minus;1]]</code> sums to <code>8 &minus; 8 = 0</code> and is the one that isolates the scratch in the four-kernel figure. The local average, by contrast, sums to 1 and reports mostly background.</li>
    </ul>
    </p>
</details>


**10. That kernel gives a strongest response of 3.129 on a defective die
and 1.118 on a clean one, with nothing trained. What does that tell you about
what training is for?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The gap is nearly a factor of three from <b>nine numbers written down by hand</b>, before any optimiser was run. Threshold that single statistic and you already have a usable grader &mdash; section 12 confirms it, where four numbers computed with this kernel put every classical family at the ceiling.</li>
        <li>So training is not what makes the hard part of this problem work. The hard part &mdash; &ldquo;respond to local contrast, ignore absolute brightness&rdquo; &mdash; was solved by <b>choosing a form of function</b>, and the form was chosen from an argument about wafers, not from data.</li>
        <li>What training then supplies is the part nobody wants to write out: which contrasts, at which scale, combined how, and with which thresholds. That is a real contribution, but it is the <i>refinement</i>, and mistaking it for the whole is how people end up throwing data at a problem whose structure they never examined.</li>
    </ul>
    </p>
</details>


**11. Of the eight 3&times;3 kernels the first layer learned, seven sum to
less than 0.6 in absolute value against a scale of 2.16. Why is that worth
reporting?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>2.16 is the scale those sums would reach if a kernel's nine weights all pointed the same way &mdash; it is the reference that makes &ldquo;less than 0.6&rdquo; mean something rather than being an unanchored small number.</li>
        <li><b>Nothing in the architecture required it.</b> There is no constraint, no penalty and no initialisation trick pushing a kernel's weights to cancel. Gradient descent arrived at near-zero sums because the data rewarded them: on dies whose background brightness varies, a kernel that responds to brightness wastes its capacity on noise.</li>
        <li>So the network <b>rediscovered section 3.3's zero-sum trick by itself</b>. That is the most encouraging thing in the lesson and also the most easily over-read: it worked here because the right answer was expressible and the data pointed at it. It is evidence that the objective was well posed, not evidence that learned features are always interpretable.</li>
    </ul>
    </p>
</details>


## Part 3 &mdash; Equivariance, pooling, and the architecture


**12. Define equivariance to translation, and explain why notebook 01's
check reports a largest difference of exactly 0 rather than merely something
small.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Equivariance</b> means shift the input and the output shifts by the same amount: <code>(shift<sub>d</sub> I) * K = shift<sub>d</sub>(I * K)</code>. The response does not change &mdash; it <i>moves</i>. Contrast <b>invariance</b>, where the output does not change at all.</li>
        <li>It follows from the definition rather than from any property of the data: the sum computed at position <code>r + d</code> of the shifted image runs over exactly the pixels the unshifted sum ran over at position <code>r</code>. The same nine multiplications, in the same order, on the same numbers.</li>
        <li>Hence <b>bit-for-bit equality</b>, not approximate agreement. The floating-point operations are identical, so their rounding is identical too. The notebook measures 0.00e+00 at four different offsets, away from the borders where a shift wraps around.</li>
        <li>The distinction matters when you read such a check: <code>8.9 &times; 10<sup>&minus;16</sup></code> was the right answer in question 6, where two different implementations summed in different orders. Here anything other than a hard zero would mean the two sides were not the same computation, and would be a bug.</li>
    </ul>
    </p>
</details>


**13. Why does a dense layer have no analogous property?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Its weights are attached to <b>pixel positions, not to patterns</b>. Shift the input by one pixel and all 576 of a unit's weights now multiply different pixels.</li>
        <li>The unit's output is therefore not shifted &mdash; it is <b>unrelated</b> to what it was. There is no operation you could apply to the dense layer's output to recover the unshifted answer, because the information about which pixel went where has already been summed away.</li>
        <li>This is the whole difference the lesson turns on, and section 6 prices it: move a defect to a band of the die where none was seen in training and the dense network drops from 0.8567 to 0.4617, while the convolutional one goes from 0.9800 to 0.9847.</li>
    </ul>
    </p>
</details>


**14. Pooling does three things. Name them, and identify the one that is a
cost rather than a benefit.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>It converts equivariance into invariance.</b> Taking a maximum over a window discards where the maximum was, so moving a defect by a pixel often leaves the pooled output literally identical.</li>
        <li><b>It enlarges the receptive field.</b> After a 2&times;2 pooling step, a 3&times;3 kernel in the next layer covers 6&times;6 of the original image for the same nine weights. Stack enough and a small kernel sees most of the die &mdash; which is how depth substitutes for width.</li>
        <li><b>It discards spatial precision</b>, and this is the cost. It is free only when <i>where</i> is not part of the answer. For a pass/fail grade it costs nothing; for segmentation, object detection or keypoint location it destroys the output you were asked for, which is why those architectures replace pooling with strides, dilation or an upsampling path.</li>
        <li>Notebook 01 pools a feature map three times, 24&times;24 down to 3&times;3, and the maximum stays 3.129 at every stage. That invariance is exactly what is wanted here and exactly what a segmentation network cannot afford.</li>
    </ul>
    </p>
</details>


**15. Why is global max pooling the design choice that makes this network's
answer independent of position, and when would it be the wrong choice?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Global max pooling takes the maximum over the <b>whole</b> feature map, returning one number per kernel: <i>did this kernel ever fire strongly, anywhere on the die?</i> Position is not reduced, it is removed.</li>
        <li>Everything after that point &mdash; the 16-unit dense head and the sigmoid output &mdash; therefore has no positional information available to depend on. The network cannot treat row 5 differently from row 15 because by then it does not know which row anything came from. That is why the moved-defect experiment costs it nothing.</li>
        <li>It is wrong whenever the answer involves <i>where</i>: locating the defect for a repair tool, counting how many defects a die carries (a maximum cannot count), or reporting their extent. It also throws away evidence when a decision depends on <b>how much</b> of the image responded rather than how strongly &mdash; average pooling is the alternative there.</li>
    </ul>
    </p>
</details>


**16. Compute the parameter count of the second convolutional layer &mdash;
16 kernels of 3&times;3 on an input of 12&times;12&times;8 &mdash; and explain why
1,168 surprises people.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>A kernel's <b>depth always matches the input's channel count</b>. Only the two spatial dimensions are yours to choose, so &ldquo;a 3&times;3 kernel&rdquo; on an 8-channel input is in fact a <code>3 &times; 3 &times; 8</code> block of 72 weights.</li>
        <li><code>16 &times; (9 &times; 8) + 16 = 16 &times; 72 + 16 = 1,152 + 16 = 1,168</code>.</li>
        <li>The expected answer is <code>16 &times; 9 + 16 = 160</code>, and the reasoning behind it is sound as far as it goes &mdash; the layer really was specified as &ldquo;3&times;3&rdquo;, and layer 1 really did cost <code>8 &times; 9 + 8 = 80</code>. What is missed is that layer 1's input had <b>one</b> channel, so its hidden factor was 1 and invisible.</li>
        <li>The consequence is worth carrying: a convolutional layer's cost is <code>(number of kernels) &times; f &times; f &times; (input channels) + (number of kernels)</code>, so it grows with the <i>product</i> of the channel counts. Doubling the channels at both ends quadruples the layer. That, not the spatial size, is what makes deep networks expensive.</li>
    </ul>
    </p>
</details>


**17. The convolutional network scores 0.9800 against the recorded grade
and 1.0000 against the true grade. Derive the first from the second, and say
what follows for anyone trying to reach 0.99.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Let <code>q</code> be the fraction of dies the model gets right against the <i>truth</i>, and <code>e</code> the probability the station records the wrong grade, independently. The model agrees with the record when both are right or both are wrong: <code>accuracy = q(1&minus;e) + (1&minus;q)e</code>.</li>
        <li>At <code>q = 1</code> and <code>e = 0.02</code> this is <code>1 &times; 0.98 + 0 &times; 0.02 = 0.9800</code> exactly, which is what was measured. The visible 2% is <b>the station's error rate in full and nothing else</b>.</li>
        <li>So there is no modelling shortfall left to close, and no architecture can do better against this label set. Effort spent reaching 0.99 would be effort spent <b>learning to reproduce the station's mistakes</b> &mdash; that is, fitting the label noise, which will not generalise because the noise is independent of the image.</li>
        <li>The whole network costs <code>80 + 1,168 + 272 + 17 = 1,537</code> parameters and reaches this, against <b>213,761</b> for lesson 9's dense network on the same images at 0.6928. Do the check: <code>576&times;256+256 = 147,712</code>, <code>256&times;256+256 = 65,792</code>, <code>256+1 = 257</code>, summing to 213,761.</li>
    </ul>
    </p>
</details>


## Part 4 &mdash; The measurement the lesson exists for


**18. Describe the moved-band experiment and its result, and explain why
the dense network lands <i>below</i> chance rather than merely lower.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Defects are confined to a band of rows: the top of the die for training, the bottom for one of the two test sets. Same generator, same defect, same contrast, same number of images &mdash; the only change is <i>where</i> the defect sits.</li>
        <li>The dense network goes <b>0.8567 &rarr; 0.4617</b>, a cost of 0.3950. The convolutional network goes 0.9800 &rarr; 0.9847, a change of +0.0047, which is within the difference between the two sets' ceilings and therefore no change at all.</li>
        <li>Below chance is the diagnostic detail. The dense network has learned &ldquo;defective means unusual values <i>in these top rows</i>&rdquo;. On the new set the top rows are always clean, so it calls almost everything acceptable &mdash; and since the two classes are near balanced, systematically answering the wrong way lands under 0.5 rather than at it. A model at exactly 0.5 has no information; one at 0.4617 has <b>anti-information</b>, evidence being used with the wrong sign.</li>
        <li>This is what an <b>inductive bias</b> is, made measurable: the convolutional network was never told where to look, because the question it asks is asked everywhere.</li>
    </ul>
    </p>
</details>


**19. Most students predict the dense network will degrade on the moved
band but stay well above chance. Explain why that prediction is reasonable and
what makes this case different.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>It is reasonable because it describes <b>almost every other kind of distribution shift</b>. Change the illumination, the sensor, the population, and performance decays smoothly; models usually retain part of what they knew. Graceful degradation is the normal experience, and expecting it is a well-calibrated prior.</li>
        <li>What is different here is that the shift is not a change of <i>degree</i> in the inputs but a change of <b>which inputs carry the signal</b>. The values in the trained rows are as they always were; the evidence has simply moved to weights that were never trained on a defect.</li>
        <li>The dense network's knowledge is stored <b>per position</b>, and the relevant positions have no knowledge in them. There is nothing to degrade gracefully &mdash; a partly-trained detector would decay, an untrained one has nothing to decay from.</li>
        <li>Carrying the right version of this matters when you generalise it: the failure is not &ldquo;dense networks are fragile&rdquo; but &ldquo;when a model stores its evidence in a coordinate the deployment changes, the loss is total, not partial&rdquo;.</li>
    </ul>
    </p>
</details>


**20. Argue that a convolutional layer is a <i>restricted</i> dense layer,
strictly less expressive, and explain why that restriction is the source of its
advantage.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>The construction.</b> Take a dense layer from the 576 inputs to the 576&times;8 outputs of the convolutional layer. Now set its weight matrix as follows: for the output at position <code>(r,c)</code> of kernel <code>k</code>, put kernel <code>k</code>'s nine weights on the nine inputs in the window at <code>(r,c)</code>, and <b>zero everywhere else</b>. The dense layer now computes exactly the convolution. Every function the convolutional layer can compute, a dense layer can compute.</li>
        <li>The converse fails: the dense layer can also set those entries freely, so it can express functions the convolution cannot &mdash; anything that treats one position differently from another. The convolutional layer is a <b>strict subset</b>, obtained by two restrictions: <i>sparsity</i> (most entries forced to zero, which is locality) and <i>tying</i> (the surviving entries forced equal across positions, which is weight sharing).</li>
        <li>So the convolution can never win on expressiveness. It wins because <b>the functions it gives up were all wrong</b>. On this problem the discarded ones are exactly those that treat row 5 differently from row 15, and nobody wanted any of them.</li>
        <li>The principle: <b>a model that cannot express a wrong answer does not have to learn to avoid it.</b> Expressiveness is what you need when you do not know the answer's shape; a restriction is what you buy when you do, and it is paid for in data you no longer have to collect.</li>
        <li>The corollary is the warning. The same restriction is a liability the moment the assumption is false &mdash; a task where position genuinely matters, such as reading a fixed-layout form, is one where the convolution's blindness has to be repaired by feeding position back in.</li>
    </ul>
    </p>
</details>


**21. Compare augmentation with architecture on this problem, using the
numbers, and say where augmentation is nonetheless indispensable.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Training the dense network on the top band only and adding randomly shifted copies takes it from <b>0.4617 to 0.5423</b>, a gain of +0.0807. The convolutional network is unmoved at 0.9847, because it had nothing to gain.</li>
        <li>Eight points is real and measurable, and still <b>forty-four points short</b> of what the architecture delivers for free. Augmentation <i>teaches</i> an invariance one example at a time, at the cost of training data and capacity, and it is approximate and can be forgotten. Architecture <i>asserts</i> one, at no cost, exactly, and it cannot be forgotten.</li>
        <li>Augmentation is indispensable for invariances <b>no layer can express</b>: brightness and contrast changes, small rotations, elastic distortion, noise, occlusion. There is no cheap architectural primitive that makes a network exactly invariant to a 7&deg; rotation, so you show it rotated copies.</li>
        <li>The rule is therefore not &ldquo;augmentation is a weak substitute&rdquo; but: use it for what no layer can assert, and do not use it to buy back an invariance a layer would have given you exactly and for nothing. <b>Translation is the canonical case of the latter.</b></li>
    </ul>
    </p>
</details>


**22. Two details of the augmentation comparison are worth copying more
than its result. Name them, and say how each could have flattered the
conclusion.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Both arms got the same number of epochs, not the same wall-clock budget.</b> The augmented set is five times larger, so equal wall-clock would have given the augmented model five times fewer passes over each image &mdash; handicapping the very arm being tested. An earlier version of this experiment did exactly that and reported a gain of 1.6 points; equalising the epochs turned it into 8.1.</li>
        <li><b>The shifts are drawn per image, not per batch.</b> One offset applied to a whole copy does not add variety &mdash; it adds four more <i>fixed</i> positions, that is, four more special cases for a per-position model to memorise. That is not augmentation, and it would have understated the gain for a different reason.</li>
        <li>The general lesson is lesson 5's: when comparing two treatments, <b>equalise everything the treatment is not</b>, and state what you equalised. Note also the direction of the first error &mdash; it produced a result the author would have been happy to believe, since it made architecture look even better. Errors that flatter your thesis are the ones you have to hunt for deliberately.</li>
    </ul>
    </p>
</details>


**23. The convolutional network's accuracy is 0.9800 at 500 training images
and still 0.9800 at 8,000. Why is that flat line not a plotting error?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Because 0.9800 <b>is the ceiling</b> of that test set. The network is already right about every die at 500 images; the only thing between it and 1.0 is the grading station, and no quantity of extra data changes the station.</li>
        <li>Say this out loud when showing the figure, because a curve that stops moving looks like a bug. The correct reading is that the learning curve has <i>terminated</i>, not stalled &mdash; there is no remaining error for more data to remove.</li>
        <li>The dense network, meanwhile, is at 0.5643 with 100 images, 0.6928 at 3,000, and 0.7455 at 8,000 &mdash; sixteen times more data than the convolutional network needed, and still <b>0.235 short</b>. The gap it is slowly closing is exactly the one weight sharing removed at the start: it is learning each position separately, so it needs enough examples per position.</li>
        <li>This is the honest way to price an inductive bias: not in accuracy at a fixed sample size, but in <b>how much data the other model needs to catch up</b> &mdash; and here the answer is more than was available.</li>
    </ul>
    </p>
</details>


## Part 5 &mdash; Which of the two assumptions is doing the work


**24. Permuting the pixels costs the convolutional network nothing on
&ldquo;is there a defect?&rdquo; and 4.5 points on &ldquo;which defect is it?&rdquo;.
State the four numbers and say which two assumptions this separates.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Detection, convolutional: <b>0.9798 as photographed, 0.9800 permuted</b> &mdash; no cost at all. Typing, convolutional: <b>0.9970 as photographed, 0.9523 permuted</b> &mdash; 4.5 points. (The dense network notices neither, on either task, exactly as lesson 9 predicted: for a dense layer a permutation is a relabelling of input units.)</li>
        <li>The usual justification &mdash; <i>images have spatial structure, convolutions exploit spatial structure</i> &mdash; bundles two separate assumptions. <b>Weight sharing:</b> a pattern means the same thing wherever it appears. <b>Locality:</b> nearby pixels belong together.</li>
        <li>A fixed permutation destroys <b>locality only</b>. Weight sharing survives it: the shuffled image is still shuffled the same way in every image, so one kernel applied everywhere is still one detector applied everywhere.</li>
        <li>Hence the split. Detection needed only weight sharing, and got the convolution's full advantage over the dense network &mdash; 0.9798 against 0.6985 &mdash; without any use of the arrangement. Typing needed locality too, and lost 4.5 points when it went.</li>
        <li>The consequence for practice: when you reach for a convolution, know <b>which of the two you are buying</b>. If it is only weight sharing, other architectures supply that as well, and the choice is open rather than settled.</li>
    </ul>
    </p>
</details>


**25. Why does grading a die survive an arbitrary permutation of its
pixels?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Because the question the task asks is <b>&ldquo;is there a patch of unusual local contrast somewhere?&rdquo;</b>, and after a permutation the defective pixels are still <i>somewhere</i>, and still extreme in value.</li>
        <li>The permutation scatters them, so they are no longer adjacent &mdash; but the kernels can learn a new arrangement-specific detector, since the permutation is <b>fixed across the whole dataset</b>. It is a relabelling of the coordinate system, not noise. A permutation drawn afresh for each image would destroy the task for every model.</li>
        <li>So the convolution is winning detection on weight sharing alone. It never needed the arrangement, and any claim of the form &ldquo;it works because images are spatially structured&rdquo; is, for this task, unsupported by its own experiment.</li>
    </ul>
    </p>
</details>


**26. Propose a way to find out, for a task of your own, which of the two
assumptions you are actually relying on &mdash; and say what you would do with
the answer.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Run the permutation test.</b> Apply one fixed random permutation to the input coordinates of every example, retrain from scratch, and compare. What survives is what weight sharing was providing; what is lost is what locality was providing. It costs one extra training run and is the only measurement here that separates the two.</li>
        <li><b>If the score barely moves</b>, locality is not doing the work, and the convolution is buying you a shared detector rather than a spatial one. Other things supply that &mdash; a tree ensemble splitting on individual features, or any model applied to a pooled summary statistic &mdash; so a convolution is a choice to justify rather than a default. The wafer detection task is exactly this case.</li>
        <li><b>If the score falls</b>, arrangement is genuinely load-bearing, and the things that follow from it become worth spending on: larger kernels, depth to grow the receptive field, and augmentations that preserve geometry. The wafer typing task is this case, and shape &mdash; a line against a blob &mdash; is what it needs.</li>
    </ul>
    </p>
</details>


## Part 6 &mdash; Transfer learning


**27. Why does this lesson pre-train on defect <i>typing</i> &mdash; a task
nobody at Meridian wants solved &mdash; rather than on the pass/fail grading task
the factory actually uses?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Because what transfers is <b>whatever the source task forced the early layers to learn</b>, and the two tasks force very different things.</li>
        <li>Grading pass/fail on scratches teaches <b>one</b> detector: dark lines. That is all the objective ever rewarded, so that is all the kernels need to encode.</li>
        <li>Naming which of three types &mdash; clean, scratch, particle &mdash; forces the early layers to <b>describe local structure in general</b>: dark lines <i>and</i> bright blobs, and enough about each to tell them apart. Those are the features worth carrying to a new defect nobody has seen.</li>
        <li>The design rule that generalises: choose a source task by asking <b>what it makes the network unable to ignore</b>, not by how similar its labels look to the target's. A harder, richer source with useless labels can be worth far more than an easy one with relevant-sounding labels.</li>
    </ul>
    </p>
</details>


**28. The pre-training task reaches 0.9993. Why does the handout call that
&ldquo;a receipt, not a result&rdquo;?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Because nobody wants a defect-type classifier. The number is not a deliverable and reporting it as an achievement would be reporting an achievement on a problem that does not exist.</li>
        <li>What it does is <b>certify the precondition</b>: the source task was actually learned, so its early layers really do encode the local structure the transfer argument assumes. Without that check, a failed transfer is ambiguous &mdash; you cannot tell whether the features were unsuitable or simply never formed.</li>
        <li>It is a habit worth copying. When an experiment depends on an intermediate step having worked, <b>measure the intermediate step and print it</b>, so that a later failure has one fewer explanation.</li>
    </ul>
    </p>
</details>


**29. The received wisdom is that transfer learning helps most when data is
scarcest. State what the measurements here show instead, with the numbers.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>At 25 images nothing works.</b> From scratch 0.5100, warm-started 0.5100 &mdash; both at chance, gain 0.0000. There is too little data to fit even a small head onto good features, so having good features buys nothing.</li>
        <li><b>Between 50 and 200 the warm start is worth 9 to 17 points</b>, and the gain <i>grows</i> across that range rather than shrinking: +0.0897 at 50 (0.6603 against 0.5707), +0.1420 at 100 (0.8640 against 0.7220), +0.1670 at 200 (0.8940 against 0.7270). This is the opposite of the slogan's shape.</li>
        <li><b>By 400 it has stopped helping</b>: 0.7877 warm-started against 0.8447 from scratch, a gain of &minus;0.0570. The from-scratch network has now seen enough of the new defect to learn its own features, and the borrowed ones have become a constraint rather than a head start.</li>
        <li>So transfer occupies a <b>window</b>, not a slope: too little data and it has nothing to attach to, too much and it has nothing to add. The practical instruction is to <b>measure the gain at your own sample size</b> rather than assuming a direction, and to re-measure it as the labelled set grows &mdash; the warm start you adopted at 100 images may be costing you at 400.</li>
    </ul>
    </p>
</details>


**30. Freezing the pre-trained base hurt even the <i>good</i> source here.
Give the numbers and explain why, and state the rule that follows.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>With the base frozen, the typing features score 0.5273 at 50 images and 0.5390 at 100, both <b>behind starting from noise</b> (0.5707 and 0.7220), and they only pull level by 200 (0.7150 against 0.7270).</li>
        <li>Yet the <b>unfrozen</b> warm start from the very same source was 0.8640 at 100 &mdash; 14 points <i>ahead</i> of from-scratch. Same weights, same data, same sample size: the only difference is whether the base was allowed to keep learning. <b>The features were useful; freezing them was the mistake.</b></li>
        <li>Why: freezing is a bet that the borrowed features are <i>already right</i> for the target. The new defect is a faint cluster of bright specks at contrast 0.24, and the typing features were tuned on stronger, differently shaped defects. Left trainable, they adapt in a few epochs from a good starting point. Frozen, whatever mismatch exists must be absorbed by the head alone, which sees only 16 numbers and cannot recover information the base failed to preserve.</li>
        <li>The rule: <b>when in doubt, warm start and leave everything trainable.</b> That recovers from a mediocre or wrong source; a frozen base cannot. Freeze only when source and target are genuinely close, or when you have so few labels that training the base would overfit &mdash; and even then, measure it rather than assume it.</li>
    </ul>
    </p>
</details>


**31. Describe the negative-transfer experiment and say why a frozen base
makes it unrecoverable.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Pre-train on scratches only &mdash; <b>dark lines</b> &mdash; and transfer, base frozen, to the new defect, which is <b>bright specks</b>. The result never leaves chance: 0.5150 at 50 images, 0.5047 at 100, 0.5350 at 200, against 0.5707 / 0.7220 / 0.7270 from scratch.</li>
        <li>The kernels are committed to reporting darkness in a place where the evidence is brightness. Roughly, they encode the wrong <i>polarity</i>, and a frozen layer <b>cannot change its mind</b> &mdash; the head downstream receives feature maps in which the new defect barely registers, and no amount of head training can recover a signal the base has already discarded.</li>
        <li>This is <b>negative transfer</b>, and the point of running it is that it is not a straw man: someone with only scratch data would reach for exactly this source, and the reasoning &ldquo;defects are defects&rdquo; sounds fine until you ask what the source objective actually rewarded.</li>
        <li>Diagnosis in practice: if a transferred model sits at chance while a from-scratch model on the same data does not, suspect the source before the hyperparameters, and unfreeze before anything else.</li>
    </ul>
    </p>
</details>


**32. Sixty labelled images of a new defect land on your desk on a Friday.
Prescribe what you would do, and what you would check later.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Sixty sits <b>inside the window</b> &mdash; above the 25 where nothing worked, in the 50-to-200 band worth 9 to 17 points here &mdash; so a warm start is the right first move rather than a hopeful one.</li>
        <li><b>Choose the source by what it forced the network to learn</b>, not by label similarity: prefer a task that required describing local structure of several kinds over one that required a single detector. If the only available source is narrow and its polarity may be wrong, expect nothing from it frozen.</li>
        <li><b>Warm start, leave every layer trainable</b>, and use a small learning rate on the base so the borrowed features adapt rather than being overwritten in the first epochs.</li>
        <li><b>Always run the from-scratch baseline alongside it.</b> It costs one training run and it is the only thing that distinguishes a useful transfer from a negative one &mdash; the frozen scratch-source arm above looked like a model that needed more epochs, not like a model that was doomed.</li>
        <li><b>Re-measure as labels accumulate.</b> The gain here had gone negative by 400 images. A warm start adopted once and never revisited becomes a constraint you are paying for without knowing it.</li>
    </ul>
    </p>
</details>


## Part 7 &mdash; Ten lessons on one problem


**33. A random forest on the raw 576 pixels reaches 0.9780, against 0.9800
for the convolutional network. What does that do to the claim that classical
methods cannot handle images?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>It <b>contradicts it</b>, on this problem, by two thousandths &mdash; and the table was built expecting to demonstrate the opposite. A support vector machine (SVM) on raw pixels reaches 0.9660 on the same data. Neither is struggling.</li>
        <li>The forest knows nothing whatever about which pixel is next to which; its 576 features are unordered as far as it is concerned. <b>It does not need to know.</b> A defect drives <i>some</i> pixel to an unusual value, and 300 trees splitting on individual pixels cover enough of them to notice.</li>
        <li>That is the forest's own <b>inductive bias</b> &mdash; &ldquo;the answer depends on a few features crossing thresholds&rdquo; &mdash; and it happens to suit this task exactly. The lesson is not that forests handle images; it is that <b>a task's requirements, not its data type, decide which assumption pays</b>. Detection here is a threshold question wearing an image's clothes.</li>
        <li>The honest caveat: this is a 24&times;24 image with one small defect and a smooth background. Scale to photographs with objects at many scales and the forest's assumption stops matching, while the convolution's still does. Report what the experiment shows and where it stops.</li>
    </ul>
    </p>
</details>


**34. k-nearest neighbours collapses to 0.4970 on raw pixels while reaching
0.9800 on four hand-made features. Explain the collapse, and name the lesson 6
result it instantiates.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>k-nearest neighbours (k-NN) has no parameters and no training; its entire model is <b>Euclidean distance between whole examples</b>. Two dies are &ldquo;similar&rdquo; if their 576 pixel values are close overall.</li>
        <li>But the distance between two dies is dominated by the <b>smooth background</b>: film thickness varies across every die, and that variation touches all 576 coordinates. A defect touches about seven of them. Its contribution to the squared distance is therefore a few parts in several hundred &mdash; <b>far smaller than the difference between two clean dies</b> that happen to have different film profiles.</li>
        <li>So the nearest neighbours of a defective die are whichever dies share its background, defect or not, and the vote is close to a coin flip. 0.4970 is chance, and the fact that it is a shade <i>below</i> 0.5 is just sampling.</li>
        <li>This is lesson 6's <b>curse of dimensionality</b>, arriving in its most concrete form: as dimension grows, distances concentrate and every point becomes roughly equidistant from every other, so &ldquo;nearest&rdquo; stops meaning &ldquo;most similar in the way that matters&rdquo;. Lesson 6 argued it with a distance-ratio plot; here it costs 48 points of accuracy.</li>
        <li>The repair is the same as lesson 6's: <b>fix the representation, not the model</b>. Given four numbers from the contrast kernel, the identical algorithm reaches the ceiling at 0.9800.</li>
    </ul>
    </p>
</details>


**35. In the synthesis table, lesson 9's dense network scores 0.7430 on raw
pixels while plain logistic regression scores 0.8905. How can adding hidden
layers make things worse?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Because <b>flexibility without a matching assumption is not an advantage; it is a larger space to search for the same answer.</b> Both models treat the 576 pixels as unordered inputs and both must learn evidence position by position &mdash; the dense network's hidden layers buy no relevant structure whatsoever on this task.</li>
        <li>What they do buy is difficulty. Logistic regression's objective is convex, so the fit is solved to optimality and its 577 parameters are as well determined as 3,000 images allow. The dense network searches a non-convex space of 213,761 parameters with the same 3,000 images, and lands on a worse point &mdash; part optimisation, part variance, and no compensating gain.</li>
        <li>This is lesson 9's own finding restated rather than contradicted. There, a hidden layer was worth <b>39 points</b> on Meridian's two-tolerance acceptance problem, where the boundary was a circle no line could enclose, and about <b>0.09</b> on the handwritten digits, where the inputs vote nearly independently. The value of a hidden layer is not a property of the layer; it is a property of the task's boundary.</li>
        <li>The practical order follows: fit the linear model first, and make anything more flexible <b>earn</b> the difference against it. On this problem the network never did.</li>
    </ul>
    </p>
</details>


**36. Four numbers computed with a hand-written kernel put every classical
family at the ceiling. State the claim the course closes on, and what it means
for choosing a model.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The four numbers are the maximum, minimum, standard deviation and largest absolute value of the contrast kernel's response over the die &mdash; computed with the nine weights written down in notebook 01 before anything was trained. On them, logistic regression, k-nearest neighbours and the support vector machine all reach <b>0.9800</b>, and the random forest 0.9795.</li>
        <li><b>A model turns a representation into a decision, and the representation usually matters more than the model.</b> Classical methods make you build it; convolutional networks learn it, paying in data and compute for the privilege. Neither is a default.</li>
        <li>The convolutional network, given only raw pixels, found an equivalent feature by itself &mdash; and the evidence that it really did is question 11's learned kernels, which sum to nearly zero exactly as the hand-written one does.</li>
        <li>For choosing: the first question is not &ldquo;which model&rdquo; but <b>&ldquo;what is the right representation, and who is going to build it&rdquo;</b>. If you can write the feature down, do that first and make everything else beat it &mdash; it is cheaper, it is inspectable, and here it was unimprovable.</li>
    </ul>
    </p>
</details>


**37. A colleague brings you three problems: a vibration trace where a
fault signature can occur at any moment; a customer table where the answer
turns on a few fields crossing thresholds; and 80 labelled examples of a rare
event with a large related dataset available. Choose an approach for each and
name the assumption you are making.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Vibration trace:</b> a one-dimensional convolution. The signature means the same thing whenever it occurs, so the assumption bought is <b>weight sharing</b>, and it is exactly the wafer detection case. Say so explicitly, because if locality across the trace turns out not to matter, a pooled summary statistic fed to a simpler model may match it &mdash; question 26's permutation test settles which.</li>
        <li><b>Customer table:</b> a tree ensemble, and it may be unbeatable. The assumption is <b>&ldquo;the answer depends on a few features crossing thresholds&rdquo;</b>, which is what a tree encodes natively. There is no geometry among the columns for a convolution to exploit, and a network here is the 0.7430-against-0.8905 situation of question 35 waiting to happen.</li>
        <li><b>80 rare examples:</b> transfer &mdash; inside the window, above the 25 where nothing works. Warm start from a source chosen for what it <i>forces</i> the network to learn, <b>leave everything trainable</b>, run the from-scratch baseline alongside, and re-measure the gain as labels accumulate.</li>
        <li>The test that covers all three: <b>if you cannot say what assumption your model encodes, you have not chosen a model, you have chosen a default.</b></li>
    </ul>
    </p>
</details>


**38. This lesson skipped batch normalisation and residual connections.
State what problem each solves, and why leaving them out was deliberate.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Real convolutional networks are deeper, and depth reintroduces lesson 9's problem: a stack of layers <b>attenuates gradients</b>, so early layers receive almost no learning signal and the network trains as though those layers were frozen at their initialisation.</li>
        <li><b>Batch normalisation</b> renormalises each layer's activations so the signal neither dies nor explodes as it propagates &mdash; lesson 9's initialisation argument, applied at every layer and at every step rather than once at the start.</li>
        <li><b>Residual connections</b> add a layer's input to its output, so the gradient has a path that <i>skips</i> the layer entirely. However badly a deep block attenuates, that shortcut carries the signal through.</li>
        <li>They were left out because <b>24&times;24 images with one small defect do not need them</b>: this network is two convolutional layers and 1,537 parameters, running on a laptop-class processor with no graphics card. A technique introduced without a problem it solves is a technique nobody remembers &mdash; and now that the problem has been named, both repairs will read as obvious when you meet them. <code>Resources/</code> carries the pointer.</li>
    </ul>
    </p>
</details>
